In [ ]:
from Bio import SeqIO
from collections import Counter
import pandas as pd 
import re
import requests
import json
import time
import glob
import pprint
import seaborn as sns
import pylab as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import ListedColormap
import pycountry 
import pycountry_convert as pc
pp = pprint.PrettyPrinter()
import numpy as np
from tabulate import tabulate
import geopandas as gpd
from geopandas import GeoDataFrame
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import regex as re
import plotly.graph_objects as go
import os

from string import punctuation
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
import nltk
from bs4 import BeautifulSoup
 

stemmer=SnowballStemmer("english")
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

import networkx as nx
from collections import defaultdict
from itertools import combinations
import sys
import ast

  


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

from pathlib import Path
import sys

# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
DATASET_CURATION_CODE_ROOT = CODE_ROOT / "dataset_curation"
FUNCTIONS_DIR = DATASET_CURATION_CODE_ROOT / "functions"
if not FUNCTIONS_DIR.exists():
    raise FileNotFoundError(
        f"Functions folder not found: {FUNCTIONS_DIR}. "
        "Copy code/dataset_curation/functions to the OneDrive code root or adjust CODE_ROOT."
    )

if str(FUNCTIONS_DIR) not in sys.path:
    sys.path.insert(0, str(FUNCTIONS_DIR))

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.

# Keep legacy string-style path variable because the downstream cells use
# expressions like path + r"processed_data\...".
path = str(DATA_ROOT) + "\\"

import backboning


### 1. merge and filter datasets

In [ ]:
folder = path+"processed_data"
substring = "scholarly_mixed"

# find matching CSVs
files_target = glob.glob(os.path.join(folder, f"*{substring}*.csv"))

# read + merge
merged_df_target = pd.concat(
    (pd.read_csv(f).assign(source_file=os.path.basename(f)) for f in files_target),
    ignore_index=True
)

df_scholarly_target = merged_df_target.drop(columns = ['Unnamed: 0','source_file']).drop_duplicates(subset = ['doi'])
df_scholarly_target_download = df_scholarly_target[(df_scholarly_target['open_access_pdf_urls'].notna()) & (df_scholarly_target['doi'].notna())]
df_scholarly_target_download.to_csv(path+r'processed_data\merged_target_mixed.csv')

In [ ]:
#here later merge the csv files of all the targets to create a unified list

folder = path+"processed_data"
substring = "merged_target"

# find matching CSVs
files = glob.glob(os.path.join(folder, f"*{substring}*.csv"))

# read + merge
merged_df = pd.concat(
    (pd.read_csv(f).assign(source_file=os.path.basename(f)) for f in files),
    ignore_index=True
)

df_scholarly = merged_df.drop(columns = ['Unnamed: 0','source_file']).drop_duplicates(subset = ['doi'])
df_scholarly_download = df_scholarly[(df_scholarly['open_access_pdf_urls'].notna()) & (df_scholarly['doi'].notna())]

df_scholarly_download.to_csv(path+r'processed_data\merged_all.csv')

In [ ]:
len(df_scholarly_download)